In [ ]:
import tensorflow as tf
tf.get_logger().setLevel('ERROR')
import logging
logging.getLogger('tensorflow').setLevel(logging.ERROR)
import warnings
warnings.filterwarnings('ignore')

### Data Loading and Preprocessing

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.regularizers import l2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.preprocessing import StandardScaler, PolynomialFeatures

# --- Load data ---
train_data = pd.read_excel("train_dataset_all_features.xlsx")
test_data_10 = pd.read_excel("2.25Cr1Mo_Test_10 (All features).xlsx")
val_data_10 = pd.read_excel("2.25Cr1Mo_Validation_10 (All features).xlsx")
                            
print(f"Train data shape: {train_data.shape}")
print(f"Test data (10) shape: {test_data_10.shape}")
print(f"Validation (10) shape: {val_data_10.shape}")

features = ['C', 'Si', 'Mn', 'P', 'S', 'Ni', 'Cr', 'Mo', 'Cu','Al', 'N', 'N Temp', 'N Time', 'T Temp', 'T Time', 'Test Stress',	'Test Temp']
X_train = train_data[features]
y_train = train_data['Lg(CL)']  

X_test_10 = test_data_10[features] 
y_test_10 = test_data_10['Lg(CL)']    

X_val_10 = val_data_10[features]
y_val_10 = val_data_10['Lg(CL)'] 

X_test = test_data[features] 
y_test = test_data['Lg(CL)']  

print(f"X_train shape: {X_train.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"X_test_10 shape: {X_test_10.shape}") 
print(f"y_test_10 shape: {y_test_10.shape}")
print(f"X_validation_10 shape: {X_val_10.shape}")
print(f"y_validation_10 shape: {y_val_10.shape}")

# Standardization 
s = StandardScaler()
X_train_s = s.fit_transform(X_train)  
X_test_10_s = s.transform(X_test_10)
X_val_10_s = s.transform(X_val_10)

print("✅ Data loaded and preprocessed successfully!")
print(f"X_train_s shape: {X_train_s.shape}")
print(f"X_test_10_s shape: {X_test_10_s.shape}")
print(f"X_validation_10_s shape: {X_val_10_s.shape}")

### Deep Narrow DNN (3 Hidden Layers)

In [ ]:
# ============================================================
# 🧠 DEEP NARROW DNN (3-LAYER) 
# ============================================================
print("🧠 IMPLEMENTING DEEP NARROW DNN (3-LAYER)")
print("="*60) 

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.regularizers import l2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.model_selection import KFold
from sklearn.model_selection import train_test_split

# ============================================================
# FLEXIBLE DNN 3-LAYER ARCHITECTURE FOR HYPERPARAMETER TUNING
# ============================================================
def create_dnn_3layer_model(input_dim, layers=[32, 24, 16], dropout_rate=0.2, 
                           l2_reg=0.001, learning_rate=0.001):
    """
    Create flexible DNN 3-layer architecture for hyperparameter tuning
    """
    model_dnn_3layer = Sequential()
    
    # Input layer
    model_dnn_3layer.add(Dense(layers[0], activation='relu', input_shape=(input_dim,),
                              kernel_initializer='he_normal',
                              kernel_regularizer=l2(l2_reg)))
    model_dnn_3layer.add(BatchNormalization())
    model_dnn_3layer.add(Dropout(dropout_rate))
    
    # Hidden layers
    for units in layers[1:]:
        model_dnn_3layer.add(Dense(units, activation='relu',
                                  kernel_initializer='he_normal',
                                  kernel_regularizer=l2(l2_reg)))
        model_dnn_3layer.add(BatchNormalization())
        model_dnn_3layer.add(Dropout(dropout_rate * 0.8))  # Slightly less dropout in deeper layers
    
    # Output layer
    model_dnn_3layer.add(Dense(1, activation='linear'))
    
    # Compile
    optimizer_dnn_3layer = Adam(learning_rate=learning_rate)
    model_dnn_3layer.compile(optimizer=optimizer_dnn_3layer, loss='mse', metrics=['mae'])
    
    return model_dnn_3layer

# ============================================================
# HYPERPARAMETER TUNING WITH K-FOLD CROSS VALIDATION
# ============================================================
print("🔍 PERFORMING HYPERPARAMETER TUNING WITH 5-FOLD CV FOR 3-LAYER DNN...")

# Define hyperparameter combinations for 3-layer DNN
hyperparam_combinations_dnn_3layer = [
    {'layers': [32, 24, 16], 'dropout_rate': 0.2, 'l2_reg': 0.001, 'learning_rate': 0.001},
    {'layers': [48, 32, 16], 'dropout_rate': 0.3, 'l2_reg': 0.002, 'learning_rate': 0.0005},
    {'layers': [24, 16, 8], 'dropout_rate': 0.1, 'l2_reg': 0.0005, 'learning_rate': 0.001},
    {'layers': [64, 32, 16], 'dropout_rate': 0.4, 'l2_reg': 0.005, 'learning_rate': 0.0001},
]

# K-Fold setup
kf_dnn_3layer = KFold(n_splits=5, shuffle=True, random_state=42)
best_params_dnn_3layer = None
best_cv_score_dnn_3layer = -float('inf')
cv_results_dnn_3layer = []

print(f"🔄 Testing {len(hyperparam_combinations_dnn_3layer)} architecture combinations for 3-layer DNN...")

for i, params_dnn_3layer in enumerate(hyperparam_combinations_dnn_3layer):
    print(f"\n🧩 Testing Combination {i+1}: {params_dnn_3layer}")
    
    fold_scores_dnn_3layer = []
    fold_predictions_dnn_3layer = np.zeros(len(y_train))
    
    for fold, (train_idx, val_idx) in enumerate(kf_dnn_3layer.split(X_train_s)):
        # Create model with current hyperparameters
        model_dnn_3layer_cv = create_dnn_3layer_model(
            input_dim=X_train_s.shape[1],
            layers=params_dnn_3layer['layers'],
            dropout_rate=params_dnn_3layer['dropout_rate'],
            l2_reg=params_dnn_3layer['l2_reg'],
            learning_rate=params_dnn_3layer['learning_rate']
        )
        
        # Callbacks
        early_stopping_dnn_3layer = EarlyStopping(
            monitor='val_loss', patience=80, restore_best_weights=True, verbose=0
        )
        reduce_lr_dnn_3layer = ReduceLROnPlateau(
            monitor='val_loss', factor=0.5, patience=40, verbose=0
        )
        
        # Train model
        history_dnn_3layer_cv = model_dnn_3layer_cv.fit(
            X_train_s[train_idx], y_train.iloc[train_idx],
            validation_data=(X_train_s[val_idx], y_train.iloc[val_idx]),
            epochs=400,
            batch_size=8,
            verbose=0,
            callbacks=[early_stopping_dnn_3layer, reduce_lr_dnn_3layer]
        )
        
        # Get predictions and score
        val_pred_dnn_3layer = model_dnn_3layer_cv.predict(X_train_s[val_idx], verbose=0).flatten()
        fold_r2_dnn_3layer = r2_score(y_train.iloc[val_idx], val_pred_dnn_3layer)
        fold_scores_dnn_3layer.append(fold_r2_dnn_3layer)
        fold_predictions_dnn_3layer[val_idx] = val_pred_dnn_3layer
        
        print(f"   Fold {fold+1} R²: {fold_r2_dnn_3layer:.4f}")
    
    # Calculate overall CV performance
    cv_r2_dnn_3layer = r2_score(y_train, fold_predictions_dnn_3layer)
    mean_fold_r2_dnn_3layer = np.mean(fold_scores_dnn_3layer)
    std_fold_r2_dnn_3layer = np.std(fold_scores_dnn_3layer)
    
    print(f"   📊 CV R²: {cv_r2_dnn_3layer:.4f}, Mean Fold R²: {mean_fold_r2_dnn_3layer:.4f} ± {std_fold_r2_dnn_3layer:.4f}")
    
    # Store results
    result_dnn_3layer = {
        'params': params_dnn_3layer,
        'cv_r2': cv_r2_dnn_3layer,
        'mean_fold_r2': mean_fold_r2_dnn_3layer,
        'std_fold_r2': std_fold_r2_dnn_3layer,
        'fold_scores': fold_scores_dnn_3layer
    }
    cv_results_dnn_3layer.append(result_dnn_3layer)
    
    # Update best parameters
    if cv_r2_dnn_3layer > best_cv_score_dnn_3layer:
        best_cv_score_dnn_3layer = cv_r2_dnn_3layer
        best_params_dnn_3layer = params_dnn_3layer
        best_cv_predictions_dnn_3layer = fold_predictions_dnn_3layer.copy()

# Display best parameters
print(f"\n🎯 BEST HYPERPARAMETERS FOR 3-LAYER DNN:")
print(f"Layers: {best_params_dnn_3layer['layers']}")
print(f"Dropout: {best_params_dnn_3layer['dropout_rate']}")
print(f"L2 Reg: {best_params_dnn_3layer['l2_reg']}")
print(f"Learning Rate: {best_params_dnn_3layer['learning_rate']}")
print(f"Best CV R²: {best_cv_score_dnn_3layer:.4f}")

# ============================================================
# TRAIN FINAL 3-LAYER MODEL WITH BEST PARAMETERS
# ============================================================
print("\n🔄 TRAINING FINAL 3-LAYER DNN WITH BEST PARAMETERS...")

# Create final model with best parameters
dnn_3layer_model = create_dnn_3layer_model(
    input_dim=X_train_s.shape[1],
    layers=best_params_dnn_3layer['layers'],
    dropout_rate=best_params_dnn_3layer['dropout_rate'],
    l2_reg=best_params_dnn_3layer['l2_reg'],
    learning_rate=best_params_dnn_3layer['learning_rate']
)

print(f"✅ FINAL 3-LAYER DNN ARCHITECTURE: 8 → {' → '.join(map(str, best_params_dnn_3layer['layers']))} → 1")
dnn_3layer_model.summary()

# Final training callbacks for 3-layer DNN
early_stopping_dnn_3layer = EarlyStopping(
    monitor='val_loss',
    patience=100,
    restore_best_weights=True,
    verbose=1,
    min_delta=0.0001
)

reduce_lr_dnn_3layer = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=50,
    min_lr=1e-7,
    verbose=1
)

# Train final 3-layer model on full training data with validation set
history_dnn_3layer = dnn_3layer_model.fit(
    X_train_s, y_train,
    validation_data=(X_val_10_s, y_val_10),
    epochs=1000,
    batch_size=8,
    verbose=1,
    callbacks=[early_stopping_dnn_3layer, reduce_lr_dnn_3layer],
    shuffle=True
)

# ============================================================
# PREDICTIONS AND METRICS FOR 3-LAYER DNN
# ============================================================
print("📊 EVALUATING FINAL 3-LAYER DNN MODEL...")

# Predictions
y_pred_log_train_dnn_3layer = dnn_3layer_model.predict(X_train_s).flatten()
y_pred_log_val_10_dnn_3layer = dnn_3layer_model.predict(X_val_10_s).flatten()
y_pred_log_test_10_dnn_3layer = dnn_3layer_model.predict(X_test_10_s).flatten()

# Metrics function for 3-layer DNN
def compute_metrics_dnn_3layer(y_true, y_pred):
    r2_dnn_3layer = r2_score(y_true, y_pred)
    mse_dnn_3layer = mean_squared_error(y_true, y_pred)
    mae_dnn_3layer = mean_absolute_error(y_true, y_pred)
    return r2_dnn_3layer, mse_dnn_3layer, mae_dnn_3layer

# Calculate metrics for 3-layer DNN
r2_dnn_train_3layer, mse_dnn_train_3layer, mae_dnn_train_3layer = compute_metrics_dnn_3layer(y_train, y_pred_log_train_dnn_3layer)
r2_dnn_val_10_3layer, mse_dnn_val_10_3layer, mae_dnn_val_10_3layer = compute_metrics_dnn_3layer(y_val_10, y_pred_log_val_10_dnn_3layer)
r2_dnn_test_10_3layer, mse_dnn_test_10_3layer, mae_dnn_test_10_3layer = compute_metrics_dnn_3layer(y_test_10, y_pred_log_test_10_dnn_3layer)

# ============================================================
# COMPREHENSIVE RESULTS FOR 3-LAYER DNN
# ============================================================
print("\n📊 3-LAYER DNN PERFORMANCE (LOG SCALE)")
print("="*60)
print(f"Training R² (log):   {r2_dnn_train_3layer:.4f}")
print(f"Validation R² (log): {r2_dnn_val_10_3layer:.4f}")
print(f"Testing R² (log):    {r2_dnn_test_10_3layer:.4f}")
print(f"Cross-Validation R² (log): {best_cv_score_dnn_3layer:.4f}")

print(f"\nCross-Validation Fold R² Scores for 3-Layer DNN:")
for i, result_dnn_3layer in enumerate(cv_results_dnn_3layer):
    if result_dnn_3layer['params'] == best_params_dnn_3layer:
        print(f"  🏆 Best 3-Layer Model Fold R²: {[f'{score:.4f}' for score in result_dnn_3layer['fold_scores']]}")

print(f"\nTraining MSE (log):  {mse_dnn_train_3layer:.4f}")
print(f"Validation MSE (log): {mse_dnn_val_10_3layer:.4f}")
print(f"Testing MSE (log):   {mse_dnn_test_10_3layer:.4f}")

print(f"\nTraining MAE (log):   {mae_dnn_train_3layer:.4f}")
print(f"Validation MAE (log): {mae_dnn_val_10_3layer:.4f}")
print(f"Testing MAE (log):    {mae_dnn_test_10_3layer:.4f}")

print(f"\nR² Difference (Train - Test): {r2_dnn_train_3layer - r2_dnn_test_10_3layer:.4f}")
print("Overfitting Indicator:", "⚠️ HIGH (Overfitting)" if (r2_dnn_train_3layer - r2_dnn_test_10_3layer) > 0.2 else "✅ GOOD")

# ============================================================
# PREDICTIONS IN BOTH SCALES FOR 3-LAYER DNN
# ============================================================
y_pred_train_dnn_3layer = np.exp(y_pred_log_train_dnn_3layer)
y_train_actual = np.exp(y_train)
y_pred_test_10_dnn_3layer = np.exp(y_pred_log_test_10_dnn_3layer)
y_test_10_actual = np.exp(y_test_10)

# First 10 predictions
print("\nFirst 10 Predictions - Test Set (LOG SCALE)")
comparison_log_df_dnn_3layer = pd.DataFrame({
    "Actual (log Rupture Time)": y_test_10[:10],
    "Predicted (log)": y_pred_log_test_10_dnn_3layer[:10],
    "Error": y_test_10[:10] - y_pred_log_test_10_dnn_3layer[:10]
}).round(4)
print(comparison_log_df_dnn_3layer)

print("\nFirst 10 Predictions - Test Set (ACTUAL SCALE)")
comparison_actual_df_dnn_3layer = pd.DataFrame({
    "Actual (Rupture Time)": y_test_10_actual[:10],
    "Predicted": y_pred_test_10_dnn_3layer[:10],
    "Error": y_test_10_actual[:10] - y_pred_test_10_dnn_3layer[:10]
}).round(2)
print(comparison_actual_df_dnn_3layer)

# ============================================================
# HYPERPARAMETER COMPARISON FOR 3-LAYER DNN
# ============================================================
print("\n🔍 3-LAYER DNN HYPERPARAMETER COMPARISON RESULTS:")
print("="*50)
for i, result_dnn_3layer in enumerate(cv_results_dnn_3layer):
    marker = "🏆" if result_dnn_3layer['params'] == best_params_dnn_3layer else "  "
    print(f"{marker} Comb {i+1}: CV R² = {result_dnn_3layer['cv_r2']:.4f}, "
          f"Layers = {result_dnn_3layer['params']['layers']}, "
          f"Dropout = {result_dnn_3layer['params']['dropout_rate']}")

# ============================================================
# TRAINING HISTORY VISUALIZATION FOR 3-LAYER DNN
# ============================================================
print("\n📈 PLOTTING 3-LAYER DNN TRAINING HISTORY...")
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history_dnn_3layer.history['loss'], label='Training Loss')
plt.plot(history_dnn_3layer.history['val_loss'], label='Validation Loss')
plt.title('3-Layer DNN: Loss Evolution')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(history_dnn_3layer.history['mae'], label='Training MAE')
plt.plot(history_dnn_3layer.history['val_mae'], label='Validation MAE')
plt.title('3-Layer DNN: MAE Evolution')
plt.xlabel('Epoch')
plt.ylabel('MAE')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n✅ DEEP NARROW DNN (3-LAYER) WITH K-FOLD CV COMPLETED!")
print("="*60)

### Plots DNN (3 Layer) 

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# -----------------------------
# Residuals
# -----------------------------
train_residuals_dnn_3layer = y_train - y_pred_log_train_dnn_3layer
test_residuals_dnn_3layer = y_test_10 - y_pred_log_test_10_dnn_3layer

# ============================================================
# 1️⃣ Predicted vs Actual (Log Scale)
# ============================================================
plt.figure(figsize=(7,6))
sns.scatterplot(x=y_train, y=y_pred_log_train_dnn_3layer, label='Train', color='steelblue', s=60)
sns.scatterplot(x=y_test_10, y=y_pred_log_test_10_dnn_3layer, label='Test', color='orange', s=60)
plt.plot([y_train.min(), y_train.max()], [y_train.min(), y_train.max()], 'k--', alpha=0.7)
plt.xlabel("Actual log(Rupture Time)")
plt.ylabel("Predicted log(Rupture Time)")
plt.title("DNN (3 layer) : Predicted vs Actual (Log Scale)")
plt.legend()
plt.tight_layout()
plt.show()

# ============================================================
# 2️⃣ Residuals vs Actual (Log Scale)
# ============================================================
plt.figure(figsize=(7,6))
sns.scatterplot(x=y_train, y=train_residuals_dnn_3layer, label='Train', color='steelblue', s=60)
sns.scatterplot(x=y_test_10, y=test_residuals_dnn_3layer, label='Test', color='orange', s=60)
plt.axhline(0, color='k', linestyle='--', alpha=0.7)
plt.xlabel("Actual log(Rupture Time)")
plt.ylabel("Residuals")
plt.title("DNN (3 layer): Residuals vs Actual (Log Scale)")
plt.legend()
plt.tight_layout()
plt.show()

# ============================================================
# 3️⃣ Histogram of Residuals (Log Scale)
# ============================================================
plt.figure(figsize=(7,6))
sns.histplot(train_residuals_dnn_3layer, color='steelblue', label='Train', kde=True, bins=30)
sns.histplot(test_residuals_dnn_3layer, color='orange', label='Test', kde=True, bins=30)
plt.xlabel("Residuals")
plt.ylabel("Count") 
plt.title("DNN (3 layer): Histogram of Residuals (Log Scale)")
plt.legend()
plt.tight_layout()
plt.show()


### ±2 SCATTER BAND ANALYSIS: DNN (3 Hidden Layers)

In [ ]:
# ============================================================
# 🔬 ±2 SCATTER BAND ANALYSIS: DNN (3 Layer)
# ============================================================
print("🔬 ANALYZING ±2 SCATTER BAND FOR DNN (3 Layer)...")
print("="*60)

# Calculate scatter ratios (Predicted/Actual)
scatter_ratio_dnn_3layer = y_pred_test_10_dnn_3layer / y_test_10_actual

# Scatter band metrics (FIXED variable names - no ± symbol)
within_2x_dnn_3layer = np.mean((scatter_ratio_dnn_3layer >= 0.5) & (scatter_ratio_dnn_3layer <= 2.0)) * 100
within_1_5x_dnn_3layer = np.mean((scatter_ratio_dnn_3layer >= 0.67) & (scatter_ratio_dnn_3layer <= 1.5)) * 100
within_3x_dnn_3layer = np.mean((scatter_ratio_dnn_3layer >= 0.33) & (scatter_ratio_dnn_3layer <= 3.0)) * 100

# Conservative vs Optimistic predictions
conservative_dnn_3layer = np.mean(scatter_ratio_dnn_3layer < 1.0) * 100  # Under-predictions
optimistic_dnn_3layer = np.mean(scatter_ratio_dnn_3layer > 1.0) * 100    # Over-predictions

print(f"📊 DNN (3 Layer) SCATTER BAND ANALYSIS:")
print(f"   ±2 Scatter Band:  {within_2x_dnn_3layer:.1f}% of predictions")
print(f"   ±1.5 Scatter Band: {within_1_5x_dnn_3layer:.1f}% of predictions") 
print(f"   ±3 Scatter Band:  {within_3x_dnn_3layer:.1f}% of predictions")
print(f"   Conservative predictions (<1.0): {conservative_dnn_3layer:.1f}%")
print(f"   Optimistic predictions (>1.0): {optimistic_dnn_3layer:.1f}%")

# ============================================================
# SCATTER BAND PLOT
# ============================================================
plt.figure(figsize=(10, 8))

# Scatter plot
plt.scatter(y_test_10_actual, y_pred_test_10_dnn_3layer, alpha=0.7, color='blue', s=50, 
           label=f'DNN (3 Layer) Predictions ({within_2x_dnn_3layer:.1f}% within ±2 band)')

# Reference lines
max_val = max(y_test_10_actual.max(), y_pred_test_10_dnn_3layer.max())
min_val = min(y_test_10_actual.min(), y_pred_test_10_dnn_3layer.min())

# Perfect prediction line
plt.plot([min_val, max_val], [min_val, max_val], 'k-', label='Perfect Prediction', linewidth=2)

# ±2 scatter band lines
plt.plot([min_val, max_val], [0.5*min_val, 0.5*max_val], 'r--', label='±2 Scatter Band', linewidth=1.5)
plt.plot([min_val, max_val], [2*min_val, 2*max_val], 'r--', linewidth=1.5)

plt.xscale('log')
plt.yscale('log')
plt.xlabel('Actual Rupture Time (hours)')
plt.ylabel('Predicted Rupture Time (hours)')
plt.title('DNN (3 Layer): ±2 Scatter Band Analysis\n(Actual Scale)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# ============================================================
# SCATTER RATIO DISTRIBUTION (IMPROVED VERSION)
# ============================================================
plt.figure(figsize=(10, 6))

# Use consistent binning and add statistics
n, bins, patches = plt.hist(scatter_ratio_dnn_3layer, bins=50, alpha=0.7, color='blue', 
                           edgecolor='black', density=False)

plt.axvline(1.0, color='red', linestyle='--', linewidth=2, label='Perfect Prediction (1.0)')
plt.axvline(0.5, color='orange', linestyle='--', linewidth=1.5, label='±2 Band Limits')
plt.axvline(2.0, color='orange', linestyle='--', linewidth=1.5)

# Add statistical annotations
median_ratio = np.median(scatter_ratio_dnn_3layer)
mean_ratio = np.mean(scatter_ratio_dnn_3layer)

plt.axvline(median_ratio, color='green', linestyle=':', linewidth=2, 
           label=f'Median Ratio: {median_ratio:.2f}')

plt.xlabel('Predicted / Actual Ratio')
plt.ylabel('Frequency')
plt.title(f'DNN (3 Layer): Scatter Ratio Distribution\n'
          f'Mean: {mean_ratio:.2f}, Median: {median_ratio:.2f}, '
          f'Within ±2: {within_2x_dnn_3layer:.1f}%')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()

# Add text box with key statistics
textstr = f'''Key Statistics:
N = {len(scatter_ratio_dnn_3layer)}
Mean = {mean_ratio:.2f}
Median = {median_ratio:.2f}
Std = {np.std(scatter_ratio_dnn_3layer):.2f}
Within ±2: {within_2x_dnn_3layer:.1f}%'''
props = dict(boxstyle='round', facecolor='wheat', alpha=0.8)
plt.gca().text(0.95, 0.95, textstr, transform=plt.gca().transAxes, 
               verticalalignment='top', horizontalalignment='right', bbox=props)

plt.show()

### Deep Narrow DNN (2 Hidden Layers)

In [ ]:
# ============================================================
# 🧠 DEEP NARROW DNN (2-LAYER) 
# ============================================================
print("🧠 IMPLEMENTING DEEP NARROW DNN (2-LAYER)")
print("="*60) 

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.regularizers import l2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.model_selection import KFold
from sklearn.model_selection import train_test_split

# ============================================================
# FLEXIBLE DNN 2-LAYER ARCHITECTURE FOR HYPERPARAMETER TUNING
# ============================================================
def create_dnn_2layer_model(input_dim, layers=[32, 16], dropout_rate=0.2, 
                           l2_reg=0.001, learning_rate=0.001):
    """
    Create flexible DNN 2-layer architecture for hyperparameter tuning
    """
    model_dnn_2layer = Sequential()
    
    # Input layer
    model_dnn_2layer.add(Dense(layers[0], activation='relu', input_shape=(input_dim,),
                              kernel_initializer='he_normal',
                              kernel_regularizer=l2(l2_reg)))
    model_dnn_2layer.add(BatchNormalization())
    model_dnn_2layer.add(Dropout(dropout_rate))
    
    # Hidden layer
    model_dnn_2layer.add(Dense(layers[1], activation='relu',
                              kernel_initializer='he_normal',
                              kernel_regularizer=l2(l2_reg)))
    model_dnn_2layer.add(BatchNormalization())
    model_dnn_2layer.add(Dropout(dropout_rate * 0.8))
    
    # Output layer
    model_dnn_2layer.add(Dense(1, activation='linear'))
    
    # Compile
    optimizer_dnn_2layer = Adam(learning_rate=learning_rate)
    model_dnn_2layer.compile(optimizer=optimizer_dnn_2layer, loss='mse', metrics=['mae'])
    
    return model_dnn_2layer

# ============================================================
# HYPERPARAMETER TUNING WITH K-FOLD CROSS VALIDATION
# ============================================================
print("🔍 PERFORMING HYPERPARAMETER TUNING WITH 5-FOLD CV FOR 2-LAYER DNN...")

# Define hyperparameter combinations for 2-layer DNN
hyperparam_combinations_dnn_2layer = [
    {'layers': [32, 16], 'dropout_rate': 0.2, 'l2_reg': 0.001, 'learning_rate': 0.001},
    {'layers': [32, 24], 'dropout_rate': 0.2, 'l2_reg': 0.001, 'learning_rate': 0.001},
    {'layers': [48, 24], 'dropout_rate': 0.3, 'l2_reg': 0.002, 'learning_rate': 0.0005},
    {'layers': [24, 12], 'dropout_rate': 0.1, 'l2_reg': 0.0005, 'learning_rate': 0.001},
    {'layers': [64, 32], 'dropout_rate': 0.4, 'l2_reg': 0.005, 'learning_rate': 0.0001},
]

# K-Fold setup
kf_dnn_2layer = KFold(n_splits=5, shuffle=True, random_state=42)
best_params_dnn_2layer = None
best_cv_score_dnn_2layer = -float('inf')
cv_results_dnn_2layer = []

print(f"🔄 Testing {len(hyperparam_combinations_dnn_2layer)} architecture combinations for 2-layer DNN...")

for i, params_dnn_2layer in enumerate(hyperparam_combinations_dnn_2layer):
    print(f"\n🧩 Testing Combination {i+1}: {params_dnn_2layer}")
    
    fold_scores_dnn_2layer = []
    fold_predictions_dnn_2layer = np.zeros(len(y_train))
    
    for fold, (train_idx, val_idx) in enumerate(kf_dnn_2layer.split(X_train_s)):
        # Create model with current hyperparameters
        model_dnn_2layer_cv = create_dnn_2layer_model(
            input_dim=X_train_s.shape[1],
            layers=params_dnn_2layer['layers'],
            dropout_rate=params_dnn_2layer['dropout_rate'],
            l2_reg=params_dnn_2layer['l2_reg'],
            learning_rate=params_dnn_2layer['learning_rate']
        )
        
        # Callbacks
        early_stopping_dnn_2layer = EarlyStopping(
            monitor='val_loss', patience=80, restore_best_weights=True, verbose=0
        )
        reduce_lr_dnn_2layer = ReduceLROnPlateau(
            monitor='val_loss', factor=0.5, patience=40, verbose=0
        )
        
        # Train model
        history_dnn_2layer_cv = model_dnn_2layer_cv.fit(
            X_train_s[train_idx], y_train.iloc[train_idx],
            validation_data=(X_train_s[val_idx], y_train.iloc[val_idx]),
            epochs=400,
            batch_size=8,
            verbose=0,
            callbacks=[early_stopping_dnn_2layer, reduce_lr_dnn_2layer]
        )
        
        # Get predictions and score
        val_pred_dnn_2layer = model_dnn_2layer_cv.predict(X_train_s[val_idx], verbose=0).flatten()
        fold_r2_dnn_2layer = r2_score(y_train.iloc[val_idx], val_pred_dnn_2layer)
        fold_scores_dnn_2layer.append(fold_r2_dnn_2layer)
        fold_predictions_dnn_2layer[val_idx] = val_pred_dnn_2layer
        
        print(f"   Fold {fold+1} R²: {fold_r2_dnn_2layer:.4f}")
    
    # Calculate overall CV performance
    cv_r2_dnn_2layer = r2_score(y_train, fold_predictions_dnn_2layer)
    mean_fold_r2_dnn_2layer = np.mean(fold_scores_dnn_2layer)
    std_fold_r2_dnn_2layer = np.std(fold_scores_dnn_2layer)
    
    print(f"   📊 CV R²: {cv_r2_dnn_2layer:.4f}, Mean Fold R²: {mean_fold_r2_dnn_2layer:.4f} ± {std_fold_r2_dnn_2layer:.4f}")
    
    # Store results
    result_dnn_2layer = {
        'params': params_dnn_2layer,
        'cv_r2': cv_r2_dnn_2layer,
        'mean_fold_r2': mean_fold_r2_dnn_2layer,
        'std_fold_r2': std_fold_r2_dnn_2layer,
        'fold_scores': fold_scores_dnn_2layer
    }
    cv_results_dnn_2layer.append(result_dnn_2layer)
    
    # Update best parameters
    if cv_r2_dnn_2layer > best_cv_score_dnn_2layer:
        best_cv_score_dnn_2layer = cv_r2_dnn_2layer
        best_params_dnn_2layer = params_dnn_2layer
        best_cv_predictions_dnn_2layer = fold_predictions_dnn_2layer.copy()

# Display best parameters
print(f"\n🎯 BEST HYPERPARAMETERS FOR 2-LAYER DNN:")
print(f"Layers: {best_params_dnn_2layer['layers']}")
print(f"Dropout: {best_params_dnn_2layer['dropout_rate']}")
print(f"L2 Reg: {best_params_dnn_2layer['l2_reg']}")
print(f"Learning Rate: {best_params_dnn_2layer['learning_rate']}")
print(f"Best CV R²: {best_cv_score_dnn_2layer:.4f}")

# ============================================================
# TRAIN FINAL 2-LAYER MODEL WITH BEST PARAMETERS
# ============================================================
print("\n🔄 TRAINING FINAL 2-LAYER DNN WITH BEST PARAMETERS...")

# Create final model with best parameters
dnn_2layer_model = create_dnn_2layer_model(
    input_dim=X_train_s.shape[1],
    layers=best_params_dnn_2layer['layers'],
    dropout_rate=best_params_dnn_2layer['dropout_rate'],
    l2_reg=best_params_dnn_2layer['l2_reg'],
    learning_rate=best_params_dnn_2layer['learning_rate']
)

print(f"✅ FINAL 2-LAYER DNN ARCHITECTURE: 8 → {' → '.join(map(str, best_params_dnn_2layer['layers']))} → 1")
dnn_2layer_model.summary()

# Final training callbacks for 2-layer DNN
early_stopping_dnn_2layer = EarlyStopping(
    monitor='val_loss',
    patience=100,
    restore_best_weights=True,
    verbose=1,
    min_delta=0.0001
)

reduce_lr_dnn_2layer = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=50,
    min_lr=1e-7,
    verbose=1
)

# Train final 2-layer model on full training data with validation set
history_dnn_2layer = dnn_2layer_model.fit(
    X_train_s, y_train,
    validation_data=(X_val_10_s, y_val_10),
    epochs=1000,
    batch_size=8,
    verbose=1,
    callbacks=[early_stopping_dnn_2layer, reduce_lr_dnn_2layer],
    shuffle=True
)

# ============================================================
# PREDICTIONS AND METRICS FOR 2-LAYER DNN
# ============================================================
print("📊 EVALUATING FINAL 2-LAYER DNN MODEL...")

# Predictions
y_pred_log_train_dnn_2layer = dnn_2layer_model.predict(X_train_s).flatten()
y_pred_log_val_10_dnn_2layer = dnn_2layer_model.predict(X_val_10_s).flatten()
y_pred_log_test_10_dnn_2layer = dnn_2layer_model.predict(X_test_10_s).flatten()

# Metrics function for 2-layer DNN
def compute_metrics_dnn_2layer(y_true, y_pred):
    r2_dnn_2layer = r2_score(y_true, y_pred)
    mse_dnn_2layer = mean_squared_error(y_true, y_pred)
    mae_dnn_2layer = mean_absolute_error(y_true, y_pred)
    return r2_dnn_2layer, mse_dnn_2layer, mae_dnn_2layer

# Calculate metrics for 2-layer DNN
r2_dnn_train_2layer, mse_dnn_train_2layer, mae_dnn_train_2layer = compute_metrics_dnn_2layer(y_train, y_pred_log_train_dnn_2layer)
r2_dnn_val_10_2layer, mse_dnn_val_10_2layer, mae_dnn_val_10_2layer = compute_metrics_dnn_2layer(y_val_10, y_pred_log_val_10_dnn_2layer)
r2_dnn_test_10_2layer, mse_dnn_test_10_2layer, mae_dnn_test_10_2layer = compute_metrics_dnn_2layer(y_test_10, y_pred_log_test_10_dnn_2layer)

# ============================================================
# COMPREHENSIVE RESULTS FOR 2-LAYER DNN
# ============================================================
print("\n📊 2-LAYER DNN PERFORMANCE (LOG SCALE)")
print("="*60)
print(f"Training R² (log):   {r2_dnn_train_2layer:.4f}")
print(f"Validation R² (log): {r2_dnn_val_10_2layer:.4f}")
print(f"Testing R² (log):    {r2_dnn_test_10_2layer:.4f}")
print(f"Cross-Validation R² (log): {best_cv_score_dnn_2layer:.4f}")

print(f"\nCross-Validation Fold R² Scores for 2-Layer DNN:")
for i, result_dnn_2layer in enumerate(cv_results_dnn_2layer):
    if result_dnn_2layer['params'] == best_params_dnn_2layer:
        print(f"  🏆 Best 2-Layer Model Fold R²: {[f'{score:.4f}' for score in result_dnn_2layer['fold_scores']]}")

print(f"\nTraining MSE (log):  {mse_dnn_train_2layer:.4f}")
print(f"Validation MSE (log): {mse_dnn_val_10_2layer:.4f}")
print(f"Testing MSE (log):   {mse_dnn_test_10_2layer:.4f}")

print(f"\nTraining MAE (log):   {mae_dnn_train_2layer:.4f}")
print(f"Validation MAE (log): {mae_dnn_val_10_2layer:.4f}")
print(f"Testing MAE (log):    {mae_dnn_test_10_2layer:.4f}")

print(f"\nR² Difference (Train - Test): {r2_dnn_train_2layer - r2_dnn_test_10_2layer:.4f}")
print("Overfitting Indicator:", "⚠️ HIGH (Overfitting)" if (r2_dnn_train_2layer - r2_dnn_test_10_2layer) > 0.2 else "✅ GOOD")

# ============================================================
# PREDICTIONS IN BOTH SCALES FOR 2-LAYER DNN
# ============================================================
y_pred_train_dnn_2layer = np.exp(y_pred_log_train_dnn_2layer)
y_train_actual = np.exp(y_train)
y_pred_test_10_dnn_2layer = np.exp(y_pred_log_test_10_dnn_2layer)
y_test_10_actual = np.exp(y_test_10)

# First 10 predictions
print("\nFirst 10 Predictions - Test Set (LOG SCALE)")
comparison_log_df_dnn_2layer = pd.DataFrame({
    "Actual (log Rupture Time)": y_test_10[:10],
    "Predicted (log)": y_pred_log_test_10_dnn_2layer[:10],
    "Error": y_test_10[:10] - y_pred_log_test_10_dnn_2layer[:10]
}).round(4)
print(comparison_log_df_dnn_2layer)

print("\nFirst 10 Predictions - Test Set (ACTUAL SCALE)")
comparison_actual_df_dnn_2layer = pd.DataFrame({
    "Actual (Rupture Time)": y_test_10_actual[:10],
    "Predicted": y_pred_test_10_dnn_2layer[:10],
    "Error": y_test_10_actual[:10] - y_pred_test_10_dnn_2layer[:10]
}).round(2)
print(comparison_actual_df_dnn_2layer)

# ============================================================
# HYPERPARAMETER COMPARISON FOR 2-LAYER DNN
# ============================================================
print("\n🔍 2-LAYER DNN HYPERPARAMETER COMPARISON RESULTS:")
print("="*50)
for i, result_dnn_2layer in enumerate(cv_results_dnn_2layer):
    marker = "🏆" if result_dnn_2layer['params'] == best_params_dnn_2layer else "  "
    print(f"{marker} Comb {i+1}: CV R² = {result_dnn_2layer['cv_r2']:.4f}, "
          f"Layers = {result_dnn_2layer['params']['layers']}, "
          f"Dropout = {result_dnn_2layer['params']['dropout_rate']}")

# ============================================================
# TRAINING HISTORY VISUALIZATION FOR 2-LAYER DNN
# ============================================================
print("\n📈 PLOTTING 2-LAYER DNN TRAINING HISTORY...")
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history_dnn_2layer.history['loss'], label='Training Loss')
plt.plot(history_dnn_2layer.history['val_loss'], label='Validation Loss')
plt.title('2-Layer DNN: Loss Evolution')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(history_dnn_2layer.history['mae'], label='Training MAE')
plt.plot(history_dnn_2layer.history['val_mae'], label='Validation MAE')
plt.title('2-Layer DNN: MAE Evolution')
plt.xlabel('Epoch')
plt.ylabel('MAE')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n✅ DEEP NARROW DNN (2-LAYER) WITH K-FOLD CV COMPLETED!")
print("="*60)

### PLots DNN (2 Layer)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# -----------------------------
# Residuals
# -----------------------------
train_residuals_dnn_2layer = y_train - y_pred_log_train_dnn_2layer
test_residuals_dnn_2layer = y_test_10 - y_pred_log_test_10_dnn_2layer

# ============================================================
# 1️⃣ Predicted vs Actual (Log Scale)
# ============================================================
plt.figure(figsize=(7,6))
sns.scatterplot(x=y_train, y=y_pred_log_train_dnn_2layer, label='Train', color='steelblue', s=60)
sns.scatterplot(x=y_test_10, y=y_pred_log_test_10_dnn_2layer, label='Test', color='orange', s=60)
plt.plot([y_train.min(), y_train.max()], [y_train.min(), y_train.max()], 'k--', alpha=0.7)
plt.xlabel("Actual log(Rupture Time)")
plt.ylabel("Predicted log(Rupture Time)")
plt.title("DNN (2 layer) : Predicted vs Actual (Log Scale)")
plt.legend()
plt.tight_layout()
plt.show()

# ============================================================
# 2️⃣ Residuals vs Actual (Log Scale)
# ============================================================
plt.figure(figsize=(7,6))
sns.scatterplot(x=y_train, y=train_residuals_dnn_2layer, label='Train', color='steelblue', s=60)
sns.scatterplot(x=y_test_10, y=test_residuals_dnn_2layer, label='Test', color='orange', s=60)
plt.axhline(0, color='k', linestyle='--', alpha=0.7)
plt.xlabel("Actual log(Rupture Time)")
plt.ylabel("Residuals")
plt.title("DNN (2 layer): Residuals vs Actual (Log Scale)")
plt.legend()
plt.tight_layout()
plt.show()

# ============================================================
# 3️⃣ Histogram of Residuals (Log Scale)
# ============================================================
plt.figure(figsize=(7,6))
sns.histplot(train_residuals_dnn_2layer, color='steelblue', label='Train', kde=True, bins=30)
sns.histplot(test_residuals_dnn_2layer, color='orange', label='Test', kde=True, bins=30)
plt.xlabel("Residuals")
plt.ylabel("Count") 
plt.title("DNN (2 layer): Histogram of Residuals (Log Scale)")
plt.legend()
plt.tight_layout()
plt.show()


### ±2 SCATTER BAND ANALYSIS: DNN (2 Hidden Layers)

In [ ]:
# ============================================================
# 🔬 ±2 SCATTER BAND ANALYSIS: DNN (2 Layer)
# ============================================================
print("🔬 ANALYZING ±2 SCATTER BAND FOR DNN (2 Layer)...")
print("="*60)

# Calculate scatter ratios (Predicted/Actual)
scatter_ratio_dnn_2layer = y_pred_test_10_dnn_2layer / y_test_10_actual

# Scatter band metrics (FIXED variable names - no ± symbol)
within_2x_dnn_2layer = np.mean((scatter_ratio_dnn_2layer >= 0.5) & (scatter_ratio_dnn_2layer <= 2.0)) * 100
within_1_5x_dnn_2layer = np.mean((scatter_ratio_dnn_2layer >= 0.67) & (scatter_ratio_dnn_2layer <= 1.5)) * 100
within_3x_dnn_2layer = np.mean((scatter_ratio_dnn_2layer >= 0.33) & (scatter_ratio_dnn_2layer <= 3.0)) * 100

# Conservative vs Optimistic predictions
conservative_dnn_2layer = np.mean(scatter_ratio_dnn_2layer < 1.0) * 100  # Under-predictions
optimistic_dnn_2layer = np.mean(scatter_ratio_dnn_2layer > 1.0) * 100    # Over-predictions

print(f"📊 DNN (2 Layer) SCATTER BAND ANALYSIS:")
print(f"   ±2 Scatter Band:  {within_2x_dnn_2layer:.1f}% of predictions")
print(f"   ±1.5 Scatter Band: {within_1_5x_dnn_2layer:.1f}% of predictions") 
print(f"   ±3 Scatter Band:  {within_3x_dnn_2layer:.1f}% of predictions")
print(f"   Conservative predictions (<1.0): {conservative_dnn_2layer:.1f}%")
print(f"   Optimistic predictions (>1.0): {optimistic_dnn_2layer:.1f}%")

# ============================================================
# SCATTER BAND PLOT
# ============================================================
plt.figure(figsize=(10, 8))

# Scatter plot
plt.scatter(y_test_10_actual, y_pred_test_10_dnn_2layer, alpha=0.7, color='blue', s=50, 
           label=f'DNN (2 Layer) Predictions ({within_2x_dnn_2layer:.1f}% within ±2 band)')

# Reference lines
max_val = max(y_test_10_actual.max(), y_pred_test_10_dnn_2layer.max())
min_val = min(y_test_10_actual.min(), y_pred_test_10_dnn_2layer.min())

# Perfect prediction line
plt.plot([min_val, max_val], [min_val, max_val], 'k-', label='Perfect Prediction', linewidth=2)

# ±2 scatter band lines
plt.plot([min_val, max_val], [0.5*min_val, 0.5*max_val], 'r--', label='±2 Scatter Band', linewidth=1.5)
plt.plot([min_val, max_val], [2*min_val, 2*max_val], 'r--', linewidth=1.5)

plt.xscale('log')
plt.yscale('log')
plt.xlabel('Actual Rupture Time (hours)')
plt.ylabel('Predicted Rupture Time (hours)')
plt.title('DNN (2 Layer): ±2 Scatter Band Analysis\n(Actual Scale)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# ============================================================
# SCATTER RATIO DISTRIBUTION (IMPROVED VERSION)
# ============================================================
plt.figure(figsize=(10, 6))

# Use consistent binning and add statistics
n, bins, patches = plt.hist(scatter_ratio_dnn_2layer, bins=50, alpha=0.7, color='blue', 
                           edgecolor='black', density=False)

plt.axvline(1.0, color='red', linestyle='--', linewidth=2, label='Perfect Prediction (1.0)')
plt.axvline(0.5, color='orange', linestyle='--', linewidth=1.5, label='±2 Band Limits')
plt.axvline(2.0, color='orange', linestyle='--', linewidth=1.5)

# Add statistical annotations
median_ratio = np.median(scatter_ratio_dnn_2layer)
mean_ratio = np.mean(scatter_ratio_dnn_2layer)

plt.axvline(median_ratio, color='green', linestyle=':', linewidth=2, 
           label=f'Median Ratio: {median_ratio:.2f}')

plt.xlabel('Predicted / Actual Ratio')
plt.ylabel('Frequency')
plt.title(f'DNN (2 Layer): Scatter Ratio Distribution\n'
          f'Mean: {mean_ratio:.2f}, Median: {median_ratio:.2f}, '
          f'Within ±2: {within_2x_dnn_2layer:.1f}%')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()

# Add text box with key statistics
textstr = f'''Key Statistics:
N = {len(scatter_ratio_dnn_2layer)}
Mean = {mean_ratio:.2f}
Median = {median_ratio:.2f}
Std = {np.std(scatter_ratio_dnn_2layer):.2f}
Within ±2: {within_2x_dnn_2layer:.1f}%'''
props = dict(boxstyle='round', facecolor='wheat', alpha=0.8)
plt.gca().text(0.95, 0.95, textstr, transform=plt.gca().transAxes, 
               verticalalignment='top', horizontalalignment='right', bbox=props)

plt.show()

### Bayesian-Style Neural Network  

In [ ]:
# ============================================================
# 🧠 BAYESIAN-STYLE NEURAL NETWORK 
# ============================================================
print("🧠 IMPLEMENTING BAYESIAN-STYLE NEURAL NETWORK WITH K-FOLD CV...")
print("="*60)

import tensorflow as tf
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.model_selection import KFold
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ============================================================
# FLEXIBLE BAYESIAN-STYLE ARCHITECTURE
# ============================================================
def create_bayesian_style_model(input_dim, layers=[32, 24, 16], dropout_rates=[0.2, 0.2, 0.1], 
                               learning_rate=0.001):
    """
    Create flexible Bayesian-style neural network with dropout for uncertainty
    """
    model = tf.keras.Sequential()
    model.add(tf.keras.layers.InputLayer(input_shape=(input_dim,)))
    
    # Add layers with specified dropout rates
    for i, (units, dropout_rate) in enumerate(zip(layers, dropout_rates)):
        model.add(tf.keras.layers.Dense(units, activation='relu'))
        model.add(tf.keras.layers.Dropout(dropout_rate))
    
    # Output layer
    model.add(tf.keras.layers.Dense(1, activation=None))
    
    optimizer = Adam(learning_rate=learning_rate)
    model.compile(optimizer=optimizer, loss='mse', metrics=['mae'])
    return model

# ============================================================
# HYPERPARAMETER TUNING WITH K-FOLD CROSS VALIDATION
# ============================================================
print("🔍 PERFORMING HYPERPARAMETER TUNING WITH 5-FOLD CV FOR BAYESIAN-STYLE NN...")

# Define hyperparameter combinations
hyperparam_combinations_bnn = [
    {'layers': [32, 24, 16], 'dropout_rates': [0.2, 0.2, 0.1], 'learning_rate': 0.001},
    {'layers': [48, 32, 16], 'dropout_rates': [0.3, 0.2, 0.1], 'learning_rate': 0.0005},
    {'layers': [24, 16, 8], 'dropout_rates': [0.1, 0.1, 0.05], 'learning_rate': 0.001},
    {'layers': [32, 24], 'dropout_rates': [0.1, 0.1], 'learning_rate': 0.001},
    {'layers': [16, 8], 'dropout_rates': [0.1, 0.1], 'learning_rate': 0.001},
]

# K-Fold setup
kf_bnn = KFold(n_splits=5, shuffle=True, random_state=42)
best_params_bnn = None
best_cv_score_bnn = -float('inf')
cv_results_bnn = []

print(f"🔄 Testing {len(hyperparam_combinations_bnn)} Bayesian-style architecture combinations...")

for i, params_bnn in enumerate(hyperparam_combinations_bnn):
    print(f"\n🧩 Testing Combination {i+1}: {params_bnn}")
    
    fold_scores_bnn = []
    fold_predictions_bnn = np.zeros(len(y_train))
    
    for fold, (train_idx, val_idx) in enumerate(kf_bnn.split(X_train_s)):
        # Create model with current hyperparameters
        model_bnn_cv = create_bayesian_style_model(
            input_dim=X_train_s.shape[1],
            layers=params_bnn['layers'],
            dropout_rates=params_bnn['dropout_rates'],
            learning_rate=params_bnn['learning_rate']
        )
        
        # Callbacks
        early_stopping_bnn = EarlyStopping(
            monitor='val_loss', patience=80, restore_best_weights=True, verbose=0
        )
        reduce_lr_bnn = ReduceLROnPlateau(
            monitor='val_loss', factor=0.5, patience=40, verbose=0
        )
        
        # Train model
        history_bnn_cv = model_bnn_cv.fit(
            X_train_s[train_idx], y_train.iloc[train_idx],
            validation_data=(X_train_s[val_idx], y_train.iloc[val_idx]),
            epochs=400,
            batch_size=8,
            verbose=0,
            callbacks=[early_stopping_bnn, reduce_lr_bnn]
        )
        
        # ============================================================
        # QUICK PREDICTION FOR CV (SPEED OPTIMIZATION)
        # ============================================================
        def predict_quick_fold(model, X):

            """
            Fast prediction for CV (no MC sampling to save time)
            """

            return model.predict(X, verbose=0).flatten()
        
        
        def predict_mc_fold(model, X, n_samples=50):

            """
            Full MC Dropout only for final evaluation
            """
            preds = np.array([model(X, training=True).numpy().flatten() for _ in range(n_samples)])
            return preds.mean(axis=0),  preds.std(axis=0)
        
        val_pred_bnn = predict_quick_fold(model_bnn_cv, X_train_s[val_idx])
        fold_r2_bnn = r2_score(y_train.iloc[val_idx], val_pred_bnn)
        fold_scores_bnn.append(fold_r2_bnn)
        fold_predictions_bnn[val_idx] = val_pred_bnn
        
        print(f"   Fold {fold+1} R²: {fold_r2_bnn:.4f}")
    
    # Calculate overall CV performance
    cv_r2_bnn = r2_score(y_train, fold_predictions_bnn)
    mean_fold_r2_bnn = np.mean(fold_scores_bnn)
    std_fold_r2_bnn = np.std(fold_scores_bnn)
    
    print(f"   📊 CV R²: {cv_r2_bnn:.4f}, Mean Fold R²: {mean_fold_r2_bnn:.4f} ± {std_fold_r2_bnn:.4f}")
    
    # Store results
    result_bnn = {
        'params': params_bnn,
        'cv_r2': cv_r2_bnn,
        'mean_fold_r2': mean_fold_r2_bnn,
        'std_fold_r2': std_fold_r2_bnn,
        'fold_scores': fold_scores_bnn
    }
    cv_results_bnn.append(result_bnn)
    
    # Update best parameters
    if cv_r2_bnn > best_cv_score_bnn:
        best_cv_score_bnn = cv_r2_bnn
        best_params_bnn = params_bnn
        best_cv_predictions_bnn = fold_predictions_bnn.copy()

# Display best parameters
print(f"\n🎯 BEST HYPERPARAMETERS FOR BAYESIAN-STYLE NN:")
print(f"Layers: {best_params_bnn['layers']}")
print(f"Dropout Rates: {best_params_bnn['dropout_rates']}")
print(f"Learning Rate: {best_params_bnn['learning_rate']}")
print(f"Best CV R²: {best_cv_score_bnn:.4f}")

# ============================================================
# TRAIN FINAL MODEL WITH BEST PARAMETERS
# ============================================================
print("\n🔄 TRAINING FINAL BAYESIAN-STYLE NN WITH BEST PARAMETERS...")

# Create final model with best parameters
bnn_model = create_bayesian_style_model(
    input_dim=X_train_s.shape[1],
    layers=best_params_bnn['layers'],
    dropout_rates=best_params_bnn['dropout_rates'],
    learning_rate=best_params_bnn['learning_rate']
)

print(f"✅ FINAL BAYESIAN-STYLE NN ARCHITECTURE: 8 → {' → '.join(map(str, best_params_bnn['layers']))} → 1")
bnn_model.summary()

# ============================================================
# FINAL TRAINING WITH UNCERTAINTY QUANTIFICATION
# ============================================================
print("\n🎯 FINAL TRAINING WITH MC DROPOUT UNCERTAINTY...")

# Callbacks for final training
early_stopping_bnn_final = EarlyStopping(
    monitor='val_loss',
    patience=100,
    restore_best_weights=True,
    verbose=1,
    min_delta=0.0001
)

reduce_lr_bnn_final = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=50,
    min_lr=1e-7,
    verbose=1
)

# Train final model
history_bnn = bnn_model.fit(
    X_train_s, y_train,
    validation_data=(X_val_10_s, y_val_10),
    epochs=1000,
    batch_size=8,
    verbose=1,
    callbacks=[early_stopping_bnn_final, reduce_lr_bnn_final],
    shuffle=True
)

# ============================================================
# MONTE CARLO DROPOUT PREDICTIONS WITH UNCERTAINTY
# ============================================================
print("📊 PERFORMING MONTE CARLO DROPOUT PREDICTIONS WITH UNCERTAINTY ESTIMATION...")

def predict_mc_dropout(model, X, n_samples=50):
    """
    Monte Carlo Dropout sampling for uncertainty estimation
    """
    preds = np.array([model(X, training=True).numpy().flatten() for _ in range(n_samples)])
    return preds.mean(axis=0), preds.std(axis=0)

# Predictions with uncertainty using MC Dropout
y_pred_log_train_bnn_mean, y_pred_log_train_bnn_std = predict_mc_dropout(bnn_model, X_train_s)
y_pred_log_val_10_bnn_mean, y_pred_log_val_10_bnn_std = predict_mc_dropout(bnn_model, X_val_10_s)
y_pred_log_test_10_bnn_mean, y_pred_log_test_10_bnn_std = predict_mc_dropout(bnn_model, X_test_10_s)

# ============================================================
# METRICS FUNCTION FOR BAYESIAN-STYLE NN
# ============================================================
def compute_metrics_bnn(y_true, y_pred):
    r2_bnn = r2_score(y_true, y_pred)
    mse_bnn = mean_squared_error(y_true, y_pred)
    mae_bnn = mean_absolute_error(y_true, y_pred)
    return r2_bnn, mse_bnn, mae_bnn

# Calculate metrics
r2_bnn_train, mse_bnn_train, mae_bnn_train = compute_metrics_bnn(y_train, y_pred_log_train_bnn_mean)
r2_bnn_val_10, mse_bnn_val_10, mae_bnn_val_10 = compute_metrics_bnn(y_val_10, y_pred_log_val_10_bnn_mean)
r2_bnn_test_10, mse_bnn_test_10, mae_bnn_test_10 = compute_metrics_bnn(y_test_10, y_pred_log_test_10_bnn_mean)

# ============================================================
# COMPREHENSIVE RESULTS
# ============================================================
print("\n📊 BAYESIAN-STYLE NEURAL NETWORK PERFORMANCE (LOG SCALE)")
print("="*60)
print(f"Training R² (log):   {r2_bnn_train:.4f}")
print(f"Validation R² (log): {r2_bnn_val_10:.4f}")
print(f"Testing R² (log):    {r2_bnn_test_10:.4f}")
print(f"Cross-Validation R² (log): {best_cv_score_bnn:.4f}")

print(f"\nCross-Validation Fold R² Scores:")
for i, result_bnn in enumerate(cv_results_bnn):
    if result_bnn['params'] == best_params_bnn:
        print(f"  🏆 Best Model Fold R²: {[f'{score:.4f}' for score in result_bnn['fold_scores']]}")

print(f"\nTraining MSE (log):  {mse_bnn_train:.4f}")
print(f"Validation MSE (log): {mse_bnn_val_10:.4f}")
print(f"Testing MSE (log):   {mse_bnn_test_10:.4f}")

print(f"\nTraining MAE (log):   {mae_bnn_train:.4f}")
print(f"Validation MAE (log): {mae_bnn_val_10:.4f}")
print(f"Testing MAE (log):    {mae_bnn_test_10:.4f}")

# Uncertainty statistics
print(f"\n📊 UNCERTAINTY QUANTIFICATION (MC Dropout):")
print(f"Training Uncertainty (std): {np.mean(y_pred_log_train_bnn_std):.4f}")
print(f"Testing Uncertainty (std):  {np.mean(y_pred_log_test_10_bnn_std):.4f}")

print(f"\nR² Difference (Train - Test): {r2_bnn_train - r2_bnn_test_10:.4f}")
print("Overfitting Indicator:", "⚠️ HIGH (Overfitting)" if (r2_bnn_train - r2_bnn_test_10) > 0.2 else "✅ GOOD")

# ============================================================
# HYPERPARAMETER COMPARISON
# ============================================================
print("\n🔍 BAYESIAN-STYLE NN HYPERPARAMETER COMPARISON RESULTS:")
print("="*50)
for i, result_bnn in enumerate(cv_results_bnn):
    marker = "🏆" if result_bnn['params'] == best_params_bnn else "  "
    print(f"{marker} Comb {i+1}: CV R² = {result_bnn['cv_r2']:.4f}, "
          f"Layers = {result_bnn['params']['layers']}, "
          f"Dropout = {result_bnn['params']['dropout_rates']}")

# ============================================================
# PREDICTIONS IN BOTH SCALES FOR BNN
# ============================================================
y_pred_train_bnn_actual = np.exp(y_pred_log_train_bnn_mean)
y_train_actual = np.exp(y_train)
y_pred_test_10_bnn_actual = np.exp(y_pred_log_test_10_bnn_mean)
y_test_10_actual = np.exp(y_test_10)

# First 10 predictions with uncertainty
print("\nFirst 10 Predictions - Test Set (LOG SCALE) with Uncertainty")
comparison_log_df_bnn = pd.DataFrame({
    "Actual (log)": y_test_10[:10],
    "Predicted (log)": y_pred_log_test_10_bnn_mean[:10],
    "Uncertainty (±)": y_pred_log_test_10_bnn_std[:10],
    "Error": y_test_10[:10] - y_pred_log_test_10_bnn_mean[:10]
}).round(4)
print(comparison_log_df_bnn)

print("\nFirst 10 Predictions - Test Set (ACTUAL SCALE)")
comparison_actual_df_bnn = pd.DataFrame({
    "Actual (Rupture Time)": y_test_10_actual[:10],
    "Predicted": y_pred_test_10_bnn_actual[:10],
    "Error": y_test_10_actual[:10] - y_pred_test_10_bnn_actual[:10]
}).round(2)
print(comparison_actual_df_bnn)

# ============================================================
# UNCERTAINTY ANALYSIS
# ============================================================
print(f"\n🔍 UNCERTAINTY ANALYSIS (MC Dropout):")
print(f"Average Prediction Uncertainty: {np.mean(y_pred_log_test_10_bnn_std):.4f}")
print(f"Max Prediction Uncertainty: {np.max(y_pred_log_test_10_bnn_std):.4f}")
print(f"Min Prediction Uncertainty: {np.min(y_pred_log_test_10_bnn_std):.4f}")

# ============================================================
# TRAINING HISTORY VISUALIZATION
# ============================================================
print("\n📈 PLOTTING TRAINING HISTORY...")
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history_bnn.history['loss'], label='Training Loss')
plt.plot(history_bnn.history['val_loss'], label='Validation Loss')
plt.title('Bayesian-Style NN: Loss Evolution')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(history_bnn.history['mae'], label='Training MAE')
plt.plot(history_bnn.history['val_mae'], label='Validation MAE')
plt.title('Bayesian-Style NN: MAE Evolution')
plt.xlabel('Epoch')
plt.ylabel('MAE')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# ============================================================
# UNCERTAINTY VISUALIZATION
# ============================================================
print("\n📊 PLOTTING PREDICTION UNCERTAINTIES...")
plt.figure(figsize=(10, 6))

# Sort by actual values for better visualization
sort_idx = np.argsort(y_test_10)
y_test_sorted = y_test_10[sort_idx]
y_pred_sorted = y_pred_log_test_10_bnn_mean[sort_idx]
y_std_sorted = y_pred_log_test_10_bnn_std[sort_idx]

plt.errorbar(range(len(y_test_sorted)), y_pred_sorted, yerr=y_std_sorted, 
             fmt='o', alpha=0.7, label='Predictions ±1σ', capsize=3)
plt.plot(range(len(y_test_sorted)), y_test_sorted, 'r-', label='Actual Values', linewidth=2)
plt.xlabel('Test Samples (Sorted)')
plt.ylabel('Log Rupture Time')
plt.title('Bayesian-Style NN: Predictions with Uncertainty Bounds')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("\n✅ BAYESIAN-STYLE NEURAL NETWORK WITH UNCERTAINTY QUANTIFICATION COMPLETED!")
print("="*60)

### 1D CONVOLUTIONAL NEURAL NETWORK (CNN)

In [ ]:
# ============================================================
# 🔍 1D CONVOLUTIONAL NEURAL NETWORK (CNN)
# ============================================================
print("🔍 IMPLEMENTING 1D CONVOLUTIONAL NEURAL NETWORK (CNN)...")
print("="*60) 

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Flatten, Dense, Dropout, BatchNormalization
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.regularizers import l2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.model_selection import KFold
from sklearn.model_selection import train_test_split

# ============================================================
# FLEXIBLE 1D CNN ARCHITECTURE FOR HYPERPARAMETER TUNING
# ============================================================
def create_cnn_model(input_dim, filters=[64, 32], kernel_sizes=[3, 3], dense_units=[32, 16], 
                    dropout_rate=0.2, l2_reg=0.001, learning_rate=0.001):
    """
    Create flexible 1D CNN architecture for hyperparameter tuning
    """
    model_cnn = Sequential()
    
    # Input layer - reshape for 1D convolution
    model_cnn.add(tf.keras.layers.InputLayer(input_shape=(input_dim, 1)))
    
    # Convolutional layers
    for i, (filt, kernel) in enumerate(zip(filters, kernel_sizes)):
        model_cnn.add(Conv1D(filters=filt, kernel_size=kernel, activation='relu',
                           kernel_initializer='he_normal',
                           kernel_regularizer=l2(l2_reg),
                           padding='same'))
        model_cnn.add(BatchNormalization())
        model_cnn.add(MaxPooling1D(pool_size=2))
        model_cnn.add(Dropout(dropout_rate))
    
    # Flatten and dense layers
    model_cnn.add(Flatten())
    
    for units in dense_units:
        model_cnn.add(Dense(units, activation='relu',
                          kernel_initializer='he_normal',
                          kernel_regularizer=l2(l2_reg)))
        model_cnn.add(BatchNormalization())
        model_cnn.add(Dropout(dropout_rate * 0.8))
    
    # Output layer
    model_cnn.add(Dense(1, activation='linear'))
    
    # Compile
    optimizer_cnn = Adam(learning_rate=learning_rate)
    model_cnn.compile(optimizer=optimizer_cnn, loss='mse', metrics=['mae'])
    
    return model_cnn

# ============================================================
# RESHAPE DATA FOR 1D CNN
# ============================================================
print("🔄 Reshaping data for 1D CNN...")
X_train_cnn = X_train_s.reshape(X_train_s.shape[0], X_train_s.shape[1], 1)
X_val_cnn = X_val_10_s.reshape(X_val_10_s.shape[0], X_val_10_s.shape[1], 1)
X_test_cnn = X_test_10_s.reshape(X_test_10_s.shape[0], X_test_10_s.shape[1], 1)

print(f"✅ CNN Data Shapes:")
print(f"X_train_cnn: {X_train_cnn.shape}")
print(f"X_val_cnn: {X_val_cnn.shape}")
print(f"X_test_cnn: {X_test_cnn.shape}")

# ============================================================
# HYPERPARAMETER TUNING WITH K-FOLD CROSS VALIDATION
# ============================================================
print("🔍 PERFORMING HYPERPARAMETER TUNING WITH 5-FOLD CV FOR CNN...")

# Define hyperparameter combinations for CNN
hyperparam_combinations_cnn = [
    # Combination 1: Higher capacity (tests upper limit)
    {'filters': [64, 32], 'kernel_sizes': [3, 3], 'dense_units': [32, 16], 
     'dropout_rate': 0.2, 'l2_reg': 0.001, 'learning_rate': 0.001},
    
    # Combination 2: Your proven [32, 24] pattern (most promising)
    {'filters': [32, 24], 'kernel_sizes': [5, 3], 'dense_units': [24, 12], 
     'dropout_rate': 0.3, 'l2_reg': 0.002, 'learning_rate': 0.0005},
    
    # Combination 3: 3-layer CNN (tests depth)
    {'filters': [32, 24, 16], 'kernel_sizes': [3, 3, 2], 'dense_units': [16, 8], 
     'dropout_rate': 0.4, 'l2_reg': 0.005, 'learning_rate': 0.0001},
    
    # Combination 4: Minimalist 2-layer
    {'filters': [24, 16], 'kernel_sizes': [3, 3], 'dense_units': [32], 
     'dropout_rate': 0.2, 'l2_reg': 0.001, 'learning_rate': 0.001},
]

# K-Fold setup
kf_cnn = KFold(n_splits=5, shuffle=True, random_state=42)
best_params_cnn = None
best_cv_score_cnn = -float('inf')
cv_results_cnn = []

print(f"🔄 Testing {len(hyperparam_combinations_cnn)} CNN architecture combinations...")

for i, params_cnn in enumerate(hyperparam_combinations_cnn):
    print(f"\n🧩 Testing CNN Combination {i+1}: {params_cnn}")
    
    fold_scores_cnn = []
    fold_predictions_cnn = np.zeros(len(y_train))
    
    for fold, (train_idx, val_idx) in enumerate(kf_cnn.split(X_train_cnn)):
        # Create model with current hyperparameters
        model_cnn_cv = create_cnn_model(
            input_dim=X_train_cnn.shape[1],
            filters=params_cnn['filters'],
            kernel_sizes=params_cnn['kernel_sizes'],
            dense_units=params_cnn['dense_units'],
            dropout_rate=params_cnn['dropout_rate'],
            l2_reg=params_cnn['l2_reg'],
            learning_rate=params_cnn['learning_rate']
        )
        
        # Callbacks
        early_stopping_cnn = EarlyStopping(
            monitor='val_loss', patience=80, restore_best_weights=True, verbose=0
        )
        reduce_lr_cnn = ReduceLROnPlateau(
            monitor='val_loss', factor=0.5, patience=40, verbose=0
        )
        
        # Train model
        history_cnn_cv = model_cnn_cv.fit(
            X_train_cnn[train_idx], y_train.iloc[train_idx],
            validation_data=(X_train_cnn[val_idx], y_train.iloc[val_idx]),
            epochs=400,
            batch_size=8,
            verbose=0,
            callbacks=[early_stopping_cnn, reduce_lr_cnn]
        )
        
        # Get predictions and score
        val_pred_cnn = model_cnn_cv.predict(X_train_cnn[val_idx], verbose=0).flatten()
        fold_r2_cnn = r2_score(y_train.iloc[val_idx], val_pred_cnn)
        fold_scores_cnn.append(fold_r2_cnn)
        fold_predictions_cnn[val_idx] = val_pred_cnn
        
        print(f"   Fold {fold+1} R²: {fold_r2_cnn:.4f}")
    
    # Calculate overall CV performance
    cv_r2_cnn = r2_score(y_train, fold_predictions_cnn)
    mean_fold_r2_cnn = np.mean(fold_scores_cnn)
    std_fold_r2_cnn = np.std(fold_scores_cnn)
    
    print(f"   📊 CV R²: {cv_r2_cnn:.4f}, Mean Fold R²: {mean_fold_r2_cnn:.4f} ± {std_fold_r2_cnn:.4f}")
    
    # Store results
    result_cnn = {
        'params': params_cnn,
        'cv_r2': cv_r2_cnn,
        'mean_fold_r2': mean_fold_r2_cnn,
        'std_fold_r2': std_fold_r2_cnn,
        'fold_scores': fold_scores_cnn
    }
    cv_results_cnn.append(result_cnn)
    
    # Update best parameters
    if cv_r2_cnn > best_cv_score_cnn:
        best_cv_score_cnn = cv_r2_cnn
        best_params_cnn = params_cnn
        best_cv_predictions_cnn = fold_predictions_cnn.copy()

# Display best parameters
print(f"\n🎯 BEST HYPERPARAMETERS FOR CNN:")
print(f"Filters: {best_params_cnn['filters']}")
print(f"Kernel Sizes: {best_params_cnn['kernel_sizes']}")
print(f"Dense Units: {best_params_cnn['dense_units']}")
print(f"Dropout: {best_params_cnn['dropout_rate']}")
print(f"L2 Reg: {best_params_cnn['l2_reg']}")
print(f"Learning Rate: {best_params_cnn['learning_rate']}")
print(f"Best CV R²: {best_cv_score_cnn:.4f}")

# ============================================================
# TRAIN FINAL CNN MODEL WITH BEST PARAMETERS
# ============================================================
print("\n🔄 TRAINING FINAL CNN WITH BEST PARAMETERS...")

# Create final model with best parameters
cnn_model = create_cnn_model(
    input_dim=X_train_cnn.shape[1],
    filters=best_params_cnn['filters'],
    kernel_sizes=best_params_cnn['kernel_sizes'],
    dense_units=best_params_cnn['dense_units'],
    dropout_rate=best_params_cnn['dropout_rate'],
    l2_reg=best_params_cnn['l2_reg'],
    learning_rate=best_params_cnn['learning_rate']
)

print(f"✅ FINAL CNN ARCHITECTURE:")
print(f"Input: {X_train_cnn.shape[1]} features → Conv1D {best_params_cnn['filters']} → Dense {best_params_cnn['dense_units']} → Output")
cnn_model.summary()

# Final training callbacks for CNN
early_stopping_cnn = EarlyStopping(
    monitor='val_loss',
    patience=100,
    restore_best_weights=True,
    verbose=1,
    min_delta=0.0001
)

reduce_lr_cnn = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=50,
    min_lr=1e-7,
    verbose=1
)

# Train final CNN model on full training data with validation set
history_cnn = cnn_model.fit(
    X_train_cnn, y_train,
    validation_data=(X_val_cnn, y_val_10),
    epochs=1000,
    batch_size=8,
    verbose=1,
    callbacks=[early_stopping_cnn, reduce_lr_cnn],
    shuffle=True
)

# ============================================================
# PREDICTIONS AND METRICS FOR CNN
# ============================================================
print("📊 EVALUATING FINAL CNN MODEL...")

# Predictions
y_pred_log_train_cnn = cnn_model.predict(X_train_cnn).flatten()
y_pred_log_val_10_cnn = cnn_model.predict(X_val_cnn).flatten()
y_pred_log_test_10_cnn = cnn_model.predict(X_test_cnn).flatten()

# Metrics function for CNN
def compute_metrics_cnn(y_true, y_pred):
    r2_cnn = r2_score(y_true, y_pred)
    mse_cnn = mean_squared_error(y_true, y_pred)
    mae_cnn = mean_absolute_error(y_true, y_pred)
    return r2_cnn, mse_cnn, mae_cnn

# Calculate metrics for CNN
r2_cnn_train, mse_cnn_train, mae_cnn_train = compute_metrics_cnn(y_train, y_pred_log_train_cnn)
r2_cnn_val_10, mse_cnn_val_10, mae_cnn_val_10 = compute_metrics_cnn(y_val_10, y_pred_log_val_10_cnn)
r2_cnn_test_10, mse_cnn_test_10, mae_cnn_test_10 = compute_metrics_cnn(y_test_10, y_pred_log_test_10_cnn)

# ============================================================
# COMPREHENSIVE RESULTS FOR CNN
# ============================================================
print("\n📊 1D CNN PERFORMANCE (LOG SCALE)")
print("="*60)
print(f"Training R² (log):   {r2_cnn_train:.4f}")
print(f"Validation R² (log): {r2_cnn_val_10:.4f}")
print(f"Testing R² (log):    {r2_cnn_test_10:.4f}")
print(f"Cross-Validation R² (log): {best_cv_score_cnn:.4f}")

print(f"\nCross-Validation Fold R² Scores for CNN:")
for i, result_cnn in enumerate(cv_results_cnn):
    if result_cnn['params'] == best_params_cnn:
        print(f"  🏆 Best CNN Model Fold R²: {[f'{score:.4f}' for score in result_cnn['fold_scores']]}")

print(f"\nTraining MSE (log):  {mse_cnn_train:.4f}")
print(f"Validation MSE (log): {mse_cnn_val_10:.4f}")
print(f"Testing MSE (log):   {mse_cnn_test_10:.4f}")

print(f"\nTraining MAE (log):   {mae_cnn_train:.4f}")
print(f"Validation MAE (log): {mae_cnn_val_10:.4f}")
print(f"Testing MAE (log):    {mae_cnn_test_10:.4f}")

print(f"\nR² Difference (Train - Test): {r2_cnn_train - r2_cnn_test_10:.4f}")
print("Overfitting Indicator:", "⚠️ HIGH (Overfitting)" if (r2_cnn_train - r2_cnn_test_10) > 0.2 else "✅ GOOD")

# ============================================================
# PREDICTIONS IN BOTH SCALES FOR CNN
# ============================================================
y_pred_train_cnn = np.exp(y_pred_log_train_cnn)
y_train_actual = np.exp(y_train)
y_pred_test_10_cnn = np.exp(y_pred_log_test_10_cnn)
y_test_10_actual = np.exp(y_test_10)

# First 10 predictions
print("\nFirst 10 Predictions - Test Set (LOG SCALE)")
comparison_log_df_cnn = pd.DataFrame({
    "Actual (log Rupture Time)": y_test_10[:10],
    "Predicted (log)": y_pred_log_test_10_cnn[:10],
    "Error": y_test_10[:10] - y_pred_log_test_10_cnn[:10]
}).round(4)
print(comparison_log_df_cnn)

print("\nFirst 10 Predictions - Test Set (ACTUAL SCALE)")
comparison_actual_df_cnn = pd.DataFrame({
    "Actual (Rupture Time)": y_test_10_actual[:10],
    "Predicted": y_pred_test_10_cnn[:10],
    "Error": y_test_10_actual[:10] - y_pred_test_10_cnn[:10]
}).round(2)
print(comparison_actual_df_cnn)

# ============================================================
# HYPERPARAMETER COMPARISON FOR CNN
# ============================================================
print("\n🔍 CNN HYPERPARAMETER COMPARISON RESULTS:")
print("="*50)
for i, result_cnn in enumerate(cv_results_cnn):
    marker = "🏆" if result_cnn['params'] == best_params_cnn else "  "
    print(f"{marker} Comb {i+1}: CV R² = {result_cnn['cv_r2']:.4f}, "
          f"Filters = {result_cnn['params']['filters']}, "
          f"Kernels = {result_cnn['params']['kernel_sizes']}")

# ============================================================
# TRAINING HISTORY VISUALIZATION FOR CNN
# ============================================================
print("\n📈 PLOTTING CNN TRAINING HISTORY...")
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history_cnn.history['loss'], label='Training Loss')
plt.plot(history_cnn.history['val_loss'], label='Validation Loss')
plt.title('1D CNN: Loss Evolution')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(history_cnn.history['mae'], label='Training MAE')
plt.plot(history_cnn.history['val_mae'], label='Validation MAE')
plt.title('1D CNN: MAE Evolution')
plt.xlabel('Epoch')
plt.ylabel('MAE')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n✅ 1D CONVOLUTIONAL NEURAL NETWORK WITH K-FOLD CV COMPLETED!")
print("="*60)

### LONG SHORT-TERM MEMORY (LSTM) NETWORK

In [ ]:
# ============================================================
# 🔄 LONG SHORT-TERM MEMORY (LSTM) NETWORK
# ============================================================
print("🔄 IMPLEMENTING LONG SHORT-TERM MEMORY (LSTM) NETWORK...")
print("="*60) 

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, BatchNormalization
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.regularizers import l2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.model_selection import KFold
from sklearn.model_selection import train_test_split

# ============================================================
# FLEXIBLE LSTM ARCHITECTURE FOR HYPERPARAMETER TUNING
# ============================================================
def create_lstm_model(input_dim, lstm_units=[64, 32], dense_units=[32, 16], 
                     dropout_rate=0.2, recurrent_dropout=0.1, l2_reg=0.001, 
                     learning_rate=0.001, return_sequences=True):
    """
    Create flexible LSTM architecture for hyperparameter tuning
    """
    model_lstm = Sequential()
    
    # Input layer - reshape for LSTM (samples, timesteps, features)
    model_lstm.add(tf.keras.layers.InputLayer(input_shape=(input_dim, 1)))
    
    # LSTM layers
    for i, units in enumerate(lstm_units):
        return_seq = (i < len(lstm_units) - 1)  # Return sequences for all but last layer
        model_lstm.add(LSTM(units=units, 
                          activation='tanh',
                          recurrent_activation='sigmoid',
                          kernel_initializer='glorot_uniform',
                          recurrent_initializer='orthogonal',
                          kernel_regularizer=l2(l2_reg),
                          recurrent_regularizer=l2(l2_reg),
                          dropout=dropout_rate,
                          recurrent_dropout=recurrent_dropout,
                          return_sequences=return_seq))
        model_lstm.add(BatchNormalization())
    
    # Dense layers
    for units in dense_units:
        model_lstm.add(Dense(units, activation='relu',
                           kernel_initializer='he_normal',
                           kernel_regularizer=l2(l2_reg)))
        model_lstm.add(BatchNormalization())
        model_lstm.add(Dropout(dropout_rate * 0.8))
    
    # Output layer
    model_lstm.add(Dense(1, activation='linear'))
    
    # Compile
    optimizer_lstm = Adam(learning_rate=learning_rate)
    model_lstm.compile(optimizer=optimizer_lstm, loss='mse', metrics=['mae'])
    
    return model_lstm

# ============================================================
# RESHAPE DATA FOR LSTM
# ============================================================
print("🔄 Reshaping data for LSTM...")
# Treat each feature as a time step (8 timesteps, 1 feature per timestep)
X_train_lstm = X_train_s.reshape(X_train_s.shape[0], X_train_s.shape[1], 1)
X_val_lstm = X_val_10_s.reshape(X_val_10_s.shape[0], X_val_10_s.shape[1], 1)
X_test_lstm = X_test_10_s.reshape(X_test_10_s.shape[0], X_test_10_s.shape[1], 1)

print(f"✅ LSTM Data Shapes:")
print(f"X_train_lstm: {X_train_lstm.shape}")  # (samples, 8 timesteps, 1 feature)
print(f"X_val_lstm: {X_val_lstm.shape}")
print(f"X_test_lstm: {X_test_lstm.shape}")

# ============================================================
# HYPERPARAMETER TUNING WITH K-FOLD CROSS VALIDATION
# ============================================================
print("🔍 PERFORMING HYPERPARAMETER TUNING WITH 5-FOLD CV FOR LSTM...")

# Define hyperparameter combinations for LSTM
hyperparam_combinations_lstm = [
    # Combination 1: Moderate 2-layer LSTM
    {'lstm_units': [64, 32], 'dense_units': [32, 16], 'dropout_rate': 0.2, 
     'recurrent_dropout': 0.1, 'l2_reg': 0.001, 'learning_rate': 0.001},
    
    # Combination 2: Your proven [32, 24] pattern adapted for LSTM
    {'lstm_units': [32, 24], 'dense_units': [24, 12], 'dropout_rate': 0.3, 
     'recurrent_dropout': 0.2, 'l2_reg': 0.002, 'learning_rate': 0.0005},
    
    # Combination 3: Lightweight 2-layer LSTM
    {'lstm_units': [32, 16], 'dense_units': [16, 8], 'dropout_rate': 0.4, 
     'recurrent_dropout': 0.3, 'l2_reg': 0.005, 'learning_rate': 0.0001},
    
    # Combination 4: Single LSTM layer (simplest)
    {'lstm_units': [64], 'dense_units': [32, 16], 'dropout_rate': 0.2, 
     'recurrent_dropout': 0.1, 'l2_reg': 0.001, 'learning_rate': 0.001},
]

# K-Fold setup
kf_lstm = KFold(n_splits=5, shuffle=True, random_state=42)
best_params_lstm = None
best_cv_score_lstm = -float('inf')
cv_results_lstm = []

print(f"🔄 Testing {len(hyperparam_combinations_lstm)} LSTM architecture combinations...")

for i, params_lstm in enumerate(hyperparam_combinations_lstm):
    print(f"\n🧩 Testing LSTM Combination {i+1}: {params_lstm}")
    
    fold_scores_lstm = []
    fold_predictions_lstm = np.zeros(len(y_train))
    
    for fold, (train_idx, val_idx) in enumerate(kf_lstm.split(X_train_lstm)):
        # Create model with current hyperparameters
        model_lstm_cv = create_lstm_model(
            input_dim=X_train_lstm.shape[1],
            lstm_units=params_lstm['lstm_units'],
            dense_units=params_lstm['dense_units'],
            dropout_rate=params_lstm['dropout_rate'],
            recurrent_dropout=params_lstm['recurrent_dropout'],
            l2_reg=params_lstm['l2_reg'],
            learning_rate=params_lstm['learning_rate']
        )
        
        # Callbacks
        early_stopping_lstm = EarlyStopping(
            monitor='val_loss', patience=80, restore_best_weights=True, verbose=0
        )
        reduce_lr_lstm = ReduceLROnPlateau(
            monitor='val_loss', factor=0.5, patience=40, verbose=0
        )
        
        # Train model
        history_lstm_cv = model_lstm_cv.fit(
            X_train_lstm[train_idx], y_train.iloc[train_idx],
            validation_data=(X_train_lstm[val_idx], y_train.iloc[val_idx]),
            epochs=400,
            batch_size=8,
            verbose=0,
            callbacks=[early_stopping_lstm, reduce_lr_lstm]
        )
        
        # Get predictions and score
        val_pred_lstm = model_lstm_cv.predict(X_train_lstm[val_idx], verbose=0).flatten()
        fold_r2_lstm = r2_score(y_train.iloc[val_idx], val_pred_lstm)
        fold_scores_lstm.append(fold_r2_lstm)
        fold_predictions_lstm[val_idx] = val_pred_lstm
        
        print(f"   Fold {fold+1} R²: {fold_r2_lstm:.4f}")
    
    # Calculate overall CV performance
    cv_r2_lstm = r2_score(y_train, fold_predictions_lstm)
    mean_fold_r2_lstm = np.mean(fold_scores_lstm)
    std_fold_r2_lstm = np.std(fold_scores_lstm)
    
    print(f"   📊 CV R²: {cv_r2_lstm:.4f}, Mean Fold R²: {mean_fold_r2_lstm:.4f} ± {std_fold_r2_lstm:.4f}")
    
    # Store results
    result_lstm = {
        'params': params_lstm,
        'cv_r2': cv_r2_lstm,
        'mean_fold_r2': mean_fold_r2_lstm,
        'std_fold_r2': std_fold_r2_lstm,
        'fold_scores': fold_scores_lstm
    }
    cv_results_lstm.append(result_lstm)
    
    # Update best parameters
    if cv_r2_lstm > best_cv_score_lstm:
        best_cv_score_lstm = cv_r2_lstm
        best_params_lstm = params_lstm
        best_cv_predictions_lstm = fold_predictions_lstm.copy()

# Display best parameters
print(f"\n🎯 BEST HYPERPARAMETERS FOR LSTM:")
print(f"LSTM Units: {best_params_lstm['lstm_units']}")
print(f"Dense Units: {best_params_lstm['dense_units']}")
print(f"Dropout: {best_params_lstm['dropout_rate']}")
print(f"Recurrent Dropout: {best_params_lstm['recurrent_dropout']}")
print(f"L2 Reg: {best_params_lstm['l2_reg']}")
print(f"Learning Rate: {best_params_lstm['learning_rate']}")
print(f"Best CV R²: {best_cv_score_lstm:.4f}")

# ============================================================
# TRAIN FINAL LSTM MODEL WITH BEST PARAMETERS
# ============================================================
print("\n🔄 TRAINING FINAL LSTM WITH BEST PARAMETERS...")

# Create final model with best parameters
lstm_model = create_lstm_model(
    input_dim=X_train_lstm.shape[1],
    lstm_units=best_params_lstm['lstm_units'],
    dense_units=best_params_lstm['dense_units'],
    dropout_rate=best_params_lstm['dropout_rate'],
    recurrent_dropout=best_params_lstm['recurrent_dropout'],
    l2_reg=best_params_lstm['l2_reg'],
    learning_rate=best_params_lstm['learning_rate']
)

print(f"✅ FINAL LSTM ARCHITECTURE:")
print(f"Input: {X_train_lstm.shape[1]} timesteps → LSTM {best_params_lstm['lstm_units']} → Dense {best_params_lstm['dense_units']} → Output")
lstm_model.summary()

# Final training callbacks for LSTM
early_stopping_lstm = EarlyStopping(
    monitor='val_loss',
    patience=100,
    restore_best_weights=True,
    verbose=1,
    min_delta=0.0001
)

reduce_lr_lstm = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=50,
    min_lr=1e-7,
    verbose=1
)

# Train final LSTM model on full training data with validation set
history_lstm = lstm_model.fit(
    X_train_lstm, y_train,
    validation_data=(X_val_lstm, y_val_10),
    epochs=1000,
    batch_size=8,
    verbose=1,
    callbacks=[early_stopping_lstm, reduce_lr_lstm],
    shuffle=True
)

# ============================================================
# PREDICTIONS AND METRICS FOR LSTM
# ============================================================
print("📊 EVALUATING FINAL LSTM MODEL...")

# Predictions
y_pred_log_train_lstm = lstm_model.predict(X_train_lstm).flatten()
y_pred_log_val_10_lstm = lstm_model.predict(X_val_lstm).flatten()
y_pred_log_test_10_lstm = lstm_model.predict(X_test_lstm).flatten()

# Metrics function for LSTM
def compute_metrics_lstm(y_true, y_pred):
    r2_lstm = r2_score(y_true, y_pred)
    mse_lstm = mean_squared_error(y_true, y_pred)
    mae_lstm = mean_absolute_error(y_true, y_pred)
    return r2_lstm, mse_lstm, mae_lstm

# Calculate metrics for LSTM
r2_lstm_train, mse_lstm_train, mae_lstm_train = compute_metrics_lstm(y_train, y_pred_log_train_lstm)
r2_lstm_val_10, mse_lstm_val_10, mae_lstm_val_10 = compute_metrics_lstm(y_val_10, y_pred_log_val_10_lstm)
r2_lstm_test_10, mse_lstm_test_10, mae_lstm_test_10 = compute_metrics_lstm(y_test_10, y_pred_log_test_10_lstm)

# ============================================================
# COMPREHENSIVE RESULTS FOR LSTM
# ============================================================
print("\n📊 LSTM PERFORMANCE (LOG SCALE)")
print("="*60)
print(f"Training R² (log):   {r2_lstm_train:.4f}")
print(f"Validation R² (log): {r2_lstm_val_10:.4f}")
print(f"Testing R² (log):    {r2_lstm_test_10:.4f}")
print(f"Cross-Validation R² (log): {best_cv_score_lstm:.4f}")

print(f"\nCross-Validation Fold R² Scores for LSTM:")
for i, result_lstm in enumerate(cv_results_lstm):
    if result_lstm['params'] == best_params_lstm:
        print(f"  🏆 Best LSTM Model Fold R²: {[f'{score:.4f}' for score in result_lstm['fold_scores']]}")

print(f"\nTraining MSE (log):  {mse_lstm_train:.4f}")
print(f"Validation MSE (log): {mse_lstm_val_10:.4f}")
print(f"Testing MSE (log):   {mse_lstm_test_10:.4f}")

print(f"\nTraining MAE (log):   {mae_lstm_train:.4f}")
print(f"Validation MAE (log): {mae_lstm_val_10:.4f}")
print(f"Testing MAE (log):    {mae_lstm_test_10:.4f}")

print(f"\nR² Difference (Train - Test): {r2_lstm_train - r2_lstm_test_10:.4f}")
print("Overfitting Indicator:", "⚠️ HIGH (Overfitting)" if (r2_lstm_train - r2_lstm_test_10) > 0.2 else "✅ GOOD")

# ============================================================
# PREDICTIONS IN BOTH SCALES FOR LSTM
# ============================================================
y_pred_train_lstm = np.exp(y_pred_log_train_lstm)
y_train_actual = np.exp(y_train)
y_pred_test_10_lstm = np.exp(y_pred_log_test_10_lstm)
y_test_10_actual = np.exp(y_test_10)

# First 10 predictions
print("\nFirst 10 Predictions - Test Set (LOG SCALE)")
comparison_log_df_lstm = pd.DataFrame({
    "Actual (log Rupture Time)": y_test_10[:10],
    "Predicted (log)": y_pred_log_test_10_lstm[:10],
    "Error": y_test_10[:10] - y_pred_log_test_10_lstm[:10]
}).round(4)
print(comparison_log_df_lstm)

print("\nFirst 10 Predictions - Test Set (ACTUAL SCALE)")
comparison_actual_df_lstm = pd.DataFrame({
    "Actual (Rupture Time)": y_test_10_actual[:10],
    "Predicted": y_pred_test_10_lstm[:10],
    "Error": y_test_10_actual[:10] - y_pred_test_10_lstm[:10]
}).round(2)
print(comparison_actual_df_lstm)

# ============================================================
# HYPERPARAMETER COMPARISON FOR LSTM
# ============================================================
print("\n🔍 LSTM HYPERPARAMETER COMPARISON RESULTS:")
print("="*50)
for i, result_lstm in enumerate(cv_results_lstm):
    marker = "🏆" if result_lstm['params'] == best_params_lstm else "  "
    print(f"{marker} Comb {i+1}: CV R² = {result_lstm['cv_r2']:.4f}, "
          f"LSTM Units = {result_lstm['params']['lstm_units']}, "
          f"Dense Units = {result_lstm['params']['dense_units']}")

# ============================================================
# TRAINING HISTORY VISUALIZATION FOR LSTM
# ============================================================
print("\n📈 PLOTTING LSTM TRAINING HISTORY...")
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history_lstm.history['loss'], label='Training Loss')
plt.plot(history_lstm.history['val_loss'], label='Validation Loss')
plt.title('LSTM: Loss Evolution')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(history_lstm.history['mae'], label='Training MAE')
plt.plot(history_lstm.history['val_mae'], label='Validation MAE')
plt.title('LSTM: MAE Evolution')
plt.xlabel('Epoch')
plt.ylabel('MAE')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n✅ LONG SHORT-TERM MEMORY NETWORK WITH K-FOLD CV COMPLETED!")
print("="*60)